# Reddit: extracción de hilos sobre donación de sangre, miedo y barreras

Este cuaderno realiza **scraping** de Reddit para obtener:
- **Posts** (hilos) relacionados con donación de sangre, miedo a donar, motivos para no donar y barreras.
- **Comentarios** de todos esos hilos (incluidas respuestas anidadas).

**Método:** Uso de la API pública de Reddit (JSON) con `User-Agent` identificado, búsqueda por subreddit + query, deduplicación por post y descarga de comentarios vía `permalink.json`. Respeto de rate limits con pausas entre peticiones.

**Salidas:** `reddit_posts_donacion_sangre.csv` y `reddit_comments_donacion_sangre.csv` (y opcionalmente versiones limpias para análisis).

In [1]:
import time
import requests
import pandas as pd

HEADERS = {
    "User-Agent": "TFG-BloodDonation-Research/1.0 (academic use; contact: 202009108@alu.comillas.edu)"
}

def reddit_search(subreddit, query, limit_total=200, sort="relevance"):
    """
    Busca en un subreddit y devuelve lista de posts (metadata).
    Usa el endpoint público /r/{subreddit}/search.json
    """
    posts = []
    after = None
    per_page = 100

    while len(posts) < limit_total:
        params = {
            "q": query,
            "restrict_sr": 1,
            "sort": sort,
            "t": "all",
            "limit": min(per_page, limit_total - len(posts)),
        }
        if after:
            params["after"] = after

        url = f"https://www.reddit.com/r/{subreddit}/search.json"
        r = requests.get(url, headers=HEADERS, params=params, timeout=30)

        if r.status_code == 429:
            print("  Rate limit (429). Esperando 60 s...")
            time.sleep(60)
            continue
        if r.status_code != 200:
            print(f"  Search failed {r.status_code}:", r.text[:150])
            break

        data = r.json()
        children = data.get("data", {}).get("children", [])
        if not children:
            break

        for c in children:
            d = c.get("data", {})
            posts.append({
                "subreddit": d.get("subreddit"),
                "id": d.get("id"),
                "title": d.get("title"),
                "selftext": d.get("selftext"),
                "created_utc": d.get("created_utc"),
                "num_comments": d.get("num_comments"),
                "score": d.get("score"),
                "permalink": "https://www.reddit.com" + (d.get("permalink") or ""),
                "url": d.get("url"),
            })

        after = data.get("data", {}).get("after")
        if not after:
            break
        time.sleep(1.2)

    return posts

def reddit_get_comments(post_permalink, limit=500):
    """
    Descarga el post + todos los comentarios (y respuestas anidadas) vía permalink.json
    """
    json_url = post_permalink.rstrip("/") + ".json"
    r = requests.get(json_url, headers=HEADERS, params={"limit": limit}, timeout=30)
    if r.status_code == 429:
        return None  # caller can retry
    if r.status_code != 200:
        return []

    payload = r.json()
    if not isinstance(payload, list) or len(payload) < 2:
        return []

    comments_listing = payload[1].get("data", {}).get("children", [])
    out = []

    def walk(nodes, depth=0):
        for n in nodes:
            if n.get("kind") != "t1":
                continue
            d = n.get("data", {})
            out.append({
                "post_permalink": post_permalink,
                "comment_id": d.get("id"),
                "body": d.get("body"),
                "created_utc": d.get("created_utc"),
                "score": d.get("score"),
                "depth": depth,
            })
            replies = d.get("replies")
            if isinstance(replies, dict):
                children = replies.get("data", {}).get("children", [])
                walk(children, depth + 1)

    walk(comments_listing, 0)
    return out

# --- Subreddits: España y preguntas generales donde aparece el tema ---
SUBREDDITS = [
    "spain", "es", "AskSpain", "madrid", "barcelona", "AskEurope", "AskReddit"
]

# --- Keywords: donación, miedo, barreras, no donar, rechazo, etc. ---
KEYWORDS = [
    '"donar sangre"', '"donación de sangre"', '"donación sangre"',
    '"miedo a donar"', '"miedo a donar sangre"', '"miedo donar sangre"',
    '"miedo a las agujas"', '"fobia a las agujas"', '"fobia agujas"',
    '"no puedo donar"', '"no puedo donar sangre"', '"no me dejan donar"',
    '"no me dejaron donar"', '"rechazado para donar"', '"me rechazaron donar"',
    '"no quiero donar"', '"no dono sangre"', '"barreras donación"',
    '"contraindicación donar"', '"contraindicado donar sangre"',
    '"mareo donando"', '"mareo donando sangre"', '"desmayo donando sangre"',
    '"requisitos para donar sangre"', '"requisitos donar sangre"',
    '"tatuaje donar sangre"', '"piercing donar sangre"',
]

LIMIT_PER_SEARCH = 80  # máx. posts por cada (subreddit, keyword)

all_posts = []
for sr in SUBREDDITS:
    for kw in KEYWORDS:
        try:
            posts = reddit_search(sr, kw, limit_total=LIMIT_PER_SEARCH, sort="relevance")
            all_posts.extend(posts)
            if posts:
                print(f"  r/{sr} '{kw[:30]}...' -> {len(posts)} posts")
        except Exception as e:
            print(f"  Error r/{sr} {kw}: {e}")
        time.sleep(1.2)

# Deduplicar por permalink
seen = set()
uniq_posts = []
for p in all_posts:
    if p.get("permalink") and p["permalink"] not in seen:
        uniq_posts.append(p)
        seen.add(p["permalink"])

print(f"\nPosts únicos: {len(uniq_posts)}")

all_comments = []
for i, p in enumerate(uniq_posts, 1):
    if not p.get("permalink"):
        continue
    cs = reddit_get_comments(p["permalink"], limit=500)
    if cs is None:
        time.sleep(60)
        cs = reddit_get_comments(p["permalink"], limit=500) or []
    all_comments.extend(cs)
    print(f"[{i}/{len(uniq_posts)}] {len(cs)} comentarios")
    time.sleep(1.2)

df_posts = pd.DataFrame(uniq_posts)
df_comments = pd.DataFrame(all_comments)

df_posts.to_csv("reddit_posts_donacion_sangre.csv", index=False)
df_comments.to_csv("reddit_comments_donacion_sangre.csv", index=False)

print(f"\nGuardado: reddit_posts_donacion_sangre.csv ({len(uniq_posts)} posts)")
print(f"         reddit_comments_donacion_sangre.csv ({len(all_comments)} comentarios)")
display(df_posts.head())
display(df_comments.head())

  r/spain '"donar sangre"...' -> 1 posts
  r/es '"donar sangre"...' -> 2 posts
  r/AskSpain '"donar sangre"...' -> 2 posts
  r/AskSpain '"donación de sangre"...' -> 1 posts
  r/AskSpain '"miedo a las agujas"...' -> 1 posts
  r/madrid '"donar sangre"...' -> 3 posts
  r/madrid '"donación de sangre"...' -> 4 posts

Posts únicos: 13
[1/13] 117 comentarios
[2/13] 243 comentarios
[3/13] 0 comentarios
[4/13] 0 comentarios
[5/13] 0 comentarios
[6/13] 0 comentarios
[7/13] 0 comentarios
[8/13] 0 comentarios
[9/13] 0 comentarios
[10/13] 10 comentarios
[11/13] 0 comentarios
[12/13] 0 comentarios
[13/13] 0 comentarios

Guardado: reddit_posts_donacion_sangre.csv (13 posts)
         reddit_comments_donacion_sangre.csv (370 comentarios)


,subreddit,id,title,selftext,created_utc,num_comments,score,permalink,url
0,spain,zmkypt,"Acudid a donar sangre quien pueda, hace falta",,1.671111e+09,118,520,https://www.reddit.com/r/spain/comments/zmkypt...,https://i.redd.it/fo8m436lt36a1.jpg
1,es,1lzwwg2,Opiniones sobre donar sangre de forma altruista,"Hola a todos,\nEl motivo de mi mensaje es que ...",1.752523e+09,246,125,https://www.reddit.com/r/es/comments/1lzwwg2/o...,https://i.redd.it/hhxqg42rbwcf1.png
2,es,34ii63,La Unión Europea NO prohíbe a los homosexuales...,,1.430486e+09,1,1,https://www.reddit.com/r/es/comments/34ii63/la...,http://estoybailando.com/la-union-europea-no-h...
3,askspain,1qhtxdb,How Can We Help? 🚂,Quiero saber si hay algo importante que pueda ...,1.768892e+09,11,11,https://www.reddit.com/r/askspain/comments/1qh...,https://www.reddit.com/r/askspain/comments/1qh...
4,askspain,1gtgprk,Donación de sangre,¿Puedo donar sangre en España siendo ciudadano...,1.731859e+09,1,1,https://www.reddit.com/r/askspain/comments/1gt...,https://www.reddit.com/r/askspain/comments/1gt...


,post_permalink,comment_id,body,created_utc,score,depth
0,https://www.reddit.com/r/spain/comments/zmkypt...,j0bii46,* **Respeta a los demás.** Debate y discute lo...,1.671111e+09,1,0
1,https://www.reddit.com/r/spain/comments/zmkypt...,j0bu7si,No soy español pero vivo aqui desde hace 3 año...,1.671116e+09,82,0
2,https://www.reddit.com/r/spain/comments/zmkypt...,j0f1xi4,Aquí te dan un bocadillo de puta madre y una c...,1.671164e+09,6,1
3,https://www.reddit.com/r/spain/comments/zmkypt...,j0fo7ls,En Málaga te dan bollería mala y zumos malos,1.671178e+09,2,2
4,https://www.reddit.com/r/spain/comments/zmkypt...,j0fswe9,Donde acudo yo (Valencia) te dan un zumito/una...,1.671182e+09,1,3


## Opcional: añadir comentarios de hilos concretos por URL

Si encuentras manualmente un hilo de Reddit sobre donación de sangre (por ejemplo buscando en Google o en Reddit), puedes pegar aquí la URL del post y descargar todos sus comentarios. Se usarán los mismos parámetros que en el notebook original: `limit=500`, `depth=10`, `raw_json=1`, y se excluyen comentarios borrados (`[deleted]`, `[removed]`). Los comentarios se añaden al CSV de comentarios existente.

In [9]:
import time
import requests
import pandas as pd

HEADERS = {
    "User-Agent": "TFG-BloodDonation-Research/1.0 (academic use; contact: 202009108@alu.comillas.edu)"
}

def get_comments(permalink, limit=500):
    """Descarga todos los comentarios de un post vía permalink.json (depth=10, raw_json=1)."""
    json_url = permalink.rstrip("/") + ".json"
    r = requests.get(
        json_url,
        headers=HEADERS,
        params={"limit": limit, "depth": 10, "raw_json": 1},
        timeout=30
    )
    print("Status:", r.status_code, "URL:", json_url[:80], "...")
    if r.status_code != 200:
        print(r.text[:200])
        return []

    # Reddit a veces devuelve HTML (rate limit, login, CAPTCHA) en lugar de JSON
    text = (r.text or "").strip()
    if not text or (text[0] != "[" and text[0] != "{"):
        print("  Reddit devolvió HTML en lugar de JSON (posible rate limit o bloqueo). Espera unos minutos.")
        return []

    try:
        payload = r.json()
    except Exception as e:
        print("  Error al parsear JSON:", e)
        return []

    if not isinstance(payload, list) or len(payload) < 2:
        return []
    comments_listing = payload[1]["data"]["children"]
    out = []

    def walk(nodes, depth=0):
        for n in nodes:
            kind = n.get("kind")
            d = n.get("data", {})

            if kind != "t1":
                continue

            body = d.get("body")
            if not body or body in ("[deleted]", "[removed]"):
                continue

            out.append({
                "post_permalink": permalink,
                "comment_id": d.get("id"),
                "body": body,
                "score": d.get("score"),
                "created_utc": d.get("created_utc"),
                "depth": depth
            })

            replies = d.get("replies")
            if isinstance(replies, dict):
                walk(replies["data"]["children"], depth + 1)

    walk(comments_listing, 0)
    return out

# Pega aquí una o varias URLs de hilos que hayas encontrado (cada una entre comillas, separadas por coma)
POST_URLS = [
    "https://www.reddit.com/r/esConversacion/comments/1lzw9hh/alguna_vez_hab%C3%A9is_donado_sangre_para_ayudar_a_la/",
    "https://www.reddit.com/r/es/comments/1lzwwg2/opiniones_sobre_donar_sangre_de_forma_altruista/",
    "https://www.reddit.com/r/AmItheAsshole/comments/ieh8nn/aita_for_refusing_to_donate_blood/?tl=es-419"

]

extra_comments = []
for url in POST_URLS:
    if not url.strip() or url.strip().startswith("#"):
        continue
    cs = get_comments(url.strip(), limit=500)
    extra_comments.extend(cs)
    print(f"  -> {len(cs)} comentarios")
    time.sleep(1.2)

if extra_comments:
    # Cargar comentarios existentes (si hay) y concatenar
    try:
        df_existing = pd.read_csv("reddit_comments_donacion_sangre.csv")
        df_extra = pd.DataFrame(extra_comments)
        df_comments = pd.concat([df_existing, df_extra], ignore_index=True)
        df_comments = df_comments.drop_duplicates(subset=["comment_id"])
    except FileNotFoundError:
        df_comments = pd.DataFrame(extra_comments)

    df_comments.to_csv("reddit_comments_donacion_sangre.csv", index=False)
    print(f"\nTotal comentarios guardados: {len(df_comments)} (reddit_comments_donacion_sangre.csv)")
    display(df_comments.tail(10))
else:
    print("No se añadieron comentarios. Comprueba las URLs en POST_URLS.")

Status: 200 URL: https://www.reddit.com/r/esConversacion/comments/1lzw9hh/alguna_vez_hab%C3%A9is_ ...
  -> 57 comentarios
Status: 200 URL: https://www.reddit.com/r/es/comments/1lzwwg2/opiniones_sobre_donar_sangre_de_for ...
  -> 243 comentarios
Status: 200 URL: https://www.reddit.com/r/AmItheAsshole/comments/ieh8nn/aita_for_refusing_to_dona ...
  Reddit devolvió HTML en lugar de JSON (posible rate limit o bloqueo). Espera unos minutos.
  -> 0 comentarios

Total comentarios guardados: 427 (reddit_comments_donacion_sangre.csv)


,post_permalink,comment_id,body,created_utc,score,depth
417,https://www.reddit.com/r/esConversacion/commen...,n3enna0,"Donaba, pero deje de hacerlo por tanta enferme...",1.752650e+09,1,0
418,https://www.reddit.com/r/esConversacion/commen...,n3gcm58,"Pues ya lo siento mucho, porque no hayan sabid...",1.752677e+09,1,1
419,https://www.reddit.com/r/esConversacion/commen...,n3f0xyy,Mi madre era donante habitual y está muy bien ...,1.752658e+09,1,0
420,https://www.reddit.com/r/esConversacion/commen...,n3gcedl,Creo que es un orgullo que tu madre haya donad...,1.752677e+09,1,1
421,https://www.reddit.com/r/esConversacion/commen...,n3shqmm,Una vez nos sacaron muestras en clase para ver...,1.752834e+09,1,0
422,https://www.reddit.com/r/esConversacion/commen...,n3uq71c,"Bueno, al menos sirvió para algo y pudisteis v...",1.752860e+09,1,1
423,https://www.reddit.com/r/esConversacion/commen...,n3559vl,"Yo paso, te roban el alma y los trigliceridos.",1.752526e+09,1,0
424,https://www.reddit.com/r/esConversacion/commen...,n357x3k,"Hombre, tanto como el alma no sé que decirte, ...",1.752526e+09,3,1
425,https://www.reddit.com/r/esConversacion/commen...,n37f9of,Estaba ironizando. Dona la sangre siempre que ...,1.752554e+09,4,2
426,https://www.reddit.com/r/esConversacion/commen...,n3gfciw,"Muchas gracias por el comentario, lo tengo en ...",1.752677e+09,1,3


## Limpieza de datos
Eliminación de nulos, duplicados, comentarios muy cortos y filtrado por términos relacionados con donación/miedo/barreras. Solo se mantienen comentarios de posts cuyo título es relevante.

In [10]:
import pandas as pd
import re

posts = pd.read_csv("reddit_posts_donacion_sangre.csv")
comments = pd.read_csv("reddit_comments_donacion_sangre.csv")

print("Posts:", posts.shape)
print("Comentarios:", comments.shape)

Posts: (13, 9)
Comentarios: (427, 6)


In [11]:
df = comments.copy()
df = df[df["body"].notna()]
df = df.drop_duplicates(subset=["body"])
df = df[df["body"].str.len() > 50]
print("Tras limpieza básica:", df.shape)

Tras limpieza básica: (372, 6)


In [12]:
terms = [
    "don", "sangre", "donar", "aguja", "agujas", "pinchazo", "miedo", "mareo", "desmayo",
    "tatu", "piercing", "anemia", "médico", "medicación", "no puedo", "no me dejan", "no dono",
    "requisito", "impiden", "rechazan", "rechazado", "contraindic", "barrera"
]
pattern = "|".join(terms)
df = df[df["body"].str.lower().str.contains(pattern)]
print("Tras filtrado semántico:", df.shape)

Tras filtrado semántico: (262, 6)


In [13]:
posts["title_clean"] = posts["title"].str.lower().fillna("")
valid_posts = posts[
    posts["title_clean"].str.contains("don|sangre|aguja|miedo|donar")
]["permalink"]
df = df[df["post_permalink"].isin(valid_posts)]
print("Tras filtrar por posts relevantes:", df.shape)

Tras filtrar por posts relevantes: (225, 6)


In [14]:
def basic_text_cleaning(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"\n", " ", text)
    text = re.sub(r"[^a-záéíóúüñ\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["clean_text"] = df["body"].apply(basic_text_cleaning)
df_clean = df[["post_permalink", "comment_id", "body", "clean_text", "score", "depth"]].copy()
df_clean.to_csv("reddit_comments_donacion_sangre_clean.csv", index=False)
print("Corpus limpio guardado: reddit_comments_donacion_sangre_clean.csv")
display(df_clean.head(10))

Corpus limpio guardado: reddit_comments_donacion_sangre_clean.csv


,post_permalink,comment_id,body,clean_text,score,depth
1,https://www.reddit.com/r/spain/comments/zmkypt...,j0bu7si,No soy español pero vivo aqui desde hace 3 año...,no soy español pero vivo aqui desde hace años ...,82,0
4,https://www.reddit.com/r/spain/comments/zmkypt...,j0fswe9,Donde acudo yo (Valencia) te dan un zumito/una...,donde acudo yo valencia te dan un zumitouna bo...,1,3
5,https://www.reddit.com/r/spain/comments/zmkypt...,j0gcqei,Oye a mi me gustan los bollos! Yo personalment...,oye a mi me gustan los bollos yo personalmente...,1,3
9,https://www.reddit.com/r/spain/comments/zmkypt...,j0egslp,No va un poco contra el espíritu de donar algo...,no va un poco contra el espíritu de donar algo...,-12,1
10,https://www.reddit.com/r/spain/comments/zmkypt...,j0f2nyk,Cómo llegas a esa conclusión?\n\nYo creo que e...,cómo llegas a esa conclusión yo creo que es al...,13,2
11,https://www.reddit.com/r/spain/comments/zmkypt...,j0ep3cl,Si donas sangre periodicamente como si quieres...,si donas sangre periodicamente como si quieres...,9,2
15,https://www.reddit.com/r/spain/comments/zmkypt...,j0bxmci,Antes de que no me dejaran donar por gordo me ...,antes de que no me dejaran donar por gordo me ...,14,1
17,https://www.reddit.com/r/spain/comments/zmkypt...,j0gcwnq,"Debería de ser al revés, donar sangre es lo qu...",debería de ser al revés donar sangre es lo que...,1,3
19,https://www.reddit.com/r/spain/comments/zmkypt...,j0boxfj,"Creo que lo del bocata ya no se hace, mi padre...",creo que lo del bocata ya no se hace mi padre ...,22,1
22,https://www.reddit.com/r/spain/comments/zmkypt...,j0c021r,"Sera eso, o incluso cambiará de centro a centr...",sera eso o incluso cambiará de centro a centro...,8,4
